In [ ]:
 # agbado study analysis 

In [1]:
import py

In [2]:
import pandas as pd        # data handling
import numpy as np         # numerical operations
import scipy.stats as stats # inferential statistics
import matplotlib.pyplot as plt  # plots

In [3]:
pwd

'C:\\Users\\ajayi'

In [4]:
cd Desktop

C:\Users\ajayi\Desktop


In [5]:
cd python_data_analysis

C:\Users\ajayi\Desktop\python_data_analysis


In [6]:
pwd

'C:\\Users\\ajayi\\Desktop\\python_data_analysis'

In [7]:
data = pd.read_excel("Mass_screening_agbado_april2026_analyze.xlsx")
 

In [8]:
data.head() 
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 193 entries, 0 to 192
Data columns (total 55 columns):
 #   Column                                          Non-Null Count  Dtype  
---  ------                                          --------------  -----  
 0   Gender                                          193 non-null    object 
 1   GenderLR                                        193 non-null    int64  
 2   Age                                             193 non-null    object 
 3   HLFE                                            165 non-null    object 
 4   Religion                                        193 non-null    object 
 5   Occupa                                          192 non-null    object 
 6   occupation                                      193 non-null    object 
 7   Marital_status                                  193 non-null    object 
 8   Family_type                                     175 non-null    object 
 9   Tribe                                      

In [9]:
df = data

In [7]:
import pandas as pd
from scipy.stats import chi2_contingency

# 1. Create the frequency table (Counts)
ct_counts = pd.crosstab(data['Family_history_of_Diabetes'], data['RBG_cat'])

# 2. Create the percentage table (Row-wise percentages)
# normalize='index' ensures percentages are calculated across each row
ct_percent = pd.crosstab(data['Family_history_of_Diabetes'], data['RBG_cat'], normalize='index') * 100

# 3. Perform the Chi-Square Test
# This function returns: chi-square stat, p-value, degrees of freedom, and expected frequencies
chi2, p, dof, expected = chi2_contingency(ct_counts)

# 4. Display Results
print("--- Percentage Table (Row %) ---")
print(ct_percent.round(2))

print("\n--- Chi-Square Test Results ---")
print(f"Chi-Square Statistic: {chi2:.4f}")
print(f"P-value: {p:.4f}")
print(f"Degrees of Freedom: {dof}")

# Quick interpretation
alpha = 0.05
if p < alpha:
    print("\nResult: Statistically significant (Reject H0)")
else:
    print("\nResult: Not statistically significant (Fail to reject H0)")

--- Percentage Table (Row %) ---
RBG_cat                     "7.8-11.0mmol/l IGT"  "<7.8mmol/l Normal"  \
Family_history_of_Diabetes                                              
Don't know                                 30.56                61.11   
No                                         28.87                66.90   
Yes                                        33.33                40.00   

RBG_cat                     ">=11.1mmol/l Diabetes"  
Family_history_of_Diabetes                           
Don't know                                     8.33  
No                                             4.23  
Yes                                           26.67  

--- Chi-Square Test Results ---
Chi-Square Statistic: 12.0065
P-value: 0.0173
Degrees of Freedom: 4

Result: Statistically significant (Reject H0)


In [9]:
# 1. Ensure the column is a categorical type with a specific order
age_order = ['18–29', '30–39', '40–49', '50–59', '60 and above']
data['Age'] = pd.Categorical(data['Age'], categories=age_order, ordered=True)

# 2. Create the dummy variables
# drop_first=True makes '18–29' your reference group automatically
age_dummies = pd.get_dummies(data['Age'], prefix='age', drop_first=True)

# 3. Add to your main dataframe for regression
data = pd.concat([data, age_dummies], axis=1)

# Now your predictors list for the regression will include:
# ['Age_30–39', 'Age_40–49', 'Age_50–59', 'Age_60 and above']


In [13]:
df = data

In [34]:
import pandas as pd
import statsmodels.api as sm
import numpy as np

# 1. Configuration
outcome = 'SBP_2lLR'
# We are using the columns that we know have data (193 rows)
predictors = ['BMI', 'GenderLR', 'famly_hx_of_htnLR', 'RBG', 'dietary_saltLR', 'tobaccoLR', 'alcoholLR']

# 2. Data Preparation
df_sub = df[[outcome] + predictors].copy()
df_sub[outcome] = pd.factorize(df_sub[outcome])[0]
df_final = pd.get_dummies(df_sub, drop_first=True)

# 3. Calculate Crude ORs (Univariate)
crude_list = []
for col in df_final.columns:
    if col == outcome: continue
    model_u = sm.Logit(df_final[outcome], sm.add_constant(df_final[col])).fit(disp=0)
    or_u = np.exp(model_u.params[1])
    ci_u = np.exp(model_u.conf_int().iloc[1])
    p_u = model_u.pvalues[1]
    crude_list.append([col, f"{or_u:.2f} ({ci_u[0]:.2f}-{ci_u[1]:.2f})", f"{p_u:.3f}"])

df_crude = pd.DataFrame(crude_list, columns=['Variable', 'Crude OR (95% CI)', 'p-value (Crude)'])

# 4. Calculate Adjusted ORs (Multivariate)
X_adj = sm.add_constant(df_final.drop(columns=[outcome]))
model_a = sm.Logit(df_final[outcome], X_adj).fit(disp=0)
or_a = np.exp(model_a.params[1:])
ci_a = np.exp(model_a.conf_int().iloc[1:])
p_a = model_a.pvalues[1:]

adj_list = []
for idx, var in enumerate(df_final.columns[1:]):
    adj_list.append([f"{or_a[idx]:.2f} ({ci_a.iloc[idx,0]:.2f}-{ci_a.iloc[idx,1]:.2f})", f"{p_a[idx]:.3f}"])

df_adj = pd.DataFrame(adj_list, columns=['Adjusted OR (95% CI)', 'p-value (Adjusted)'])

# 5. Merge and Display
final_table = pd.concat([df_crude, df_adj], axis=1)
print("--- JOURNAL READY LOGISTIC REGRESSION TABLE ---")
display(final_table)

C:\Users\ajayi\AppData\Local\Temp\ipykernel_4872\3100638832.py:20: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  or_u = np.exp(model_u.params[1])
C:\Users\ajayi\AppData\Local\Temp\ipykernel_4872\3100638832.py:22: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  p_u = model_u.pvalues[1]
C:\Users\ajayi\AppData\Local\Temp\ipykernel_4872\3100638832.py:20: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  or_u = np.exp(model_u.params[1])
C:\Use

--- JOURNAL READY LOGISTIC REGRESSION TABLE ---


,Variable,Crude OR (95% CI),p-value (Crude),Adjusted OR (95% CI),p-value (Adjusted)
0,BMI,0.98 (0.92-1.04),0.440,0.97 (0.90-1.03),0.308
1,GenderLR,0.60 (0.30-1.17),0.135,0.65 (0.24-1.71),0.378
2,famly_hx_of_htnLR,0.68 (0.19-2.43),0.554,0.73 (0.20-2.71),0.637
3,RBG,1.01 (0.91-1.12),0.836,1.01 (0.90-1.13),0.855
4,dietary_saltLR,0.87 (0.44-1.70),0.681,0.90 (0.45-1.79),0.761
5,tobaccoLR,0.87 (0.29-2.62),0.799,1.18 (0.34-4.07),0.789
6,alcoholLR,0.52 (0.25-1.06),0.072,0.69 (0.26-1.84),0.462


In [36]:
# This creates a separate column for each age group automatically
X_age = pd.get_dummies(df['Age'], drop_first=True) 
X_age = sm.add_constant(X_age)
y = pd.factorize(df['SBP_2lLR'])[0]

model = sm.Logit(y, X_age).fit()
print(model.summary())

Optimization terminated successfully.
         Current function value: 0.597590
         Iterations 5
                           Logit Regression Results                           
Dep. Variable:                      y   No. Observations:                  193
Model:                          Logit   Df Residuals:                      192
Method:                           MLE   Df Model:                            0
Date:                Mon, 11 May 2026   Pseudo R-squ.:               4.927e-11
Time:                        02:46:48   Log-Likelihood:                -115.33
converged:                       True   LL-Null:                       -115.33
Covariance Type:            nonrobust   LLR p-value:                       nan
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.9199      0.159      5.769      0.000       0.607       1.232


In [41]:
import pandas as pd
import statsmodels.api as sm
import numpy as np

# 1. Setup the outcome (SBP_21LR)
y = pd.factorize(df['SBP_2lLR'])[0]

# 2. Setup Predictors - We handle Age separately to ensure it works
age_dummies = pd.get_dummies(df['Age'], prefix='Age', drop_first=True)
other_cols = ['BMI', 'GenderLR', 'RBG', 'famly_hx_of_htnLR', 
              'dietary_saltLR', 'tobaccoLR', 'alcoholLR']

# Combine age groups with other data and convert text to numbers
X_all = pd.concat([age_dummies, df[other_cols]], axis=1)
X_all = pd.get_dummies(X_all, drop_first=True).astype(float)

# Fill any missing values with the median so the "zero-size" error disappears
X_all = X_all.fillna(X_all.median())

# 3. Calculate Crude (Unadjusted) ORs
crude_data = []
for col in X_all.columns:
    X_u = sm.add_constant(X_all[col])
    model_u = sm.Logit(y, X_u).fit(disp=0)
    
    or_u = np.exp(model_u.params[1])
    ci_u = np.exp(model_u.conf_int().iloc[1])
    p_u = model_u.pvalues[1]
    crude_data.append([col, f"{or_u:.2f} ({ci_u[0]:.2f}-{ci_u[1]:.2f})", f"{p_u:.3f}"])

# 4. Calculate Adjusted (Multivariate) ORs
X_adj = sm.add_constant(X_all)
model_a = sm.Logit(y, X_adj).fit(disp=0)
or_a = np.exp(model_a.params[1:])
ci_a = np.exp(model_a.conf_int().iloc[1:])
p_a = model_a.pvalues[1:]

# 5. Merge into Final Table
df_final_results = pd.DataFrame(crude_data, columns=['Variable', 'Crude OR (95% CI)', 'p-value (Crude)'])
adj_list = [f"{or_a[i]:.2f} ({ci_a.iloc[i,0]:.2f}-{ci_a.iloc[i,1]:.2f})" for i in range(len(or_a))]
adj_p_list = [f"{p_a[i]:.3f}" for i in range(len(p_a))]

df_final_results['Adjusted OR (95% CI)'] = adj_list
df_final_results['p-value (Adjusted)'] = adj_p_list

print("--- FINAL PUBLICATION TABLE ---")
display(df_final_results)

C:\Users\ajayi\AppData\Local\Temp\ipykernel_4872\791116979.py:26: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  or_u = np.exp(model_u.params[1])
C:\Users\ajayi\AppData\Local\Temp\ipykernel_4872\791116979.py:28: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  p_u = model_u.pvalues[1]
C:\Users\ajayi\AppData\Local\Temp\ipykernel_4872\791116979.py:26: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  or_u = np.exp(model_u.params[1])
C:\Users\

--- FINAL PUBLICATION TABLE ---


,Variable,Crude OR (95% CI),p-value (Crude),Adjusted OR (95% CI),p-value (Adjusted)
0,BMI,0.98 (0.92-1.04),0.440,0.97 (0.90-1.03),0.308
1,GenderLR,0.60 (0.30-1.17),0.135,0.65 (0.24-1.71),0.378
2,RBG,1.01 (0.91-1.12),0.836,1.01 (0.90-1.13),0.855
3,famly_hx_of_htnLR,0.68 (0.19-2.43),0.554,0.73 (0.20-2.71),0.637
4,dietary_saltLR,0.87 (0.44-1.70),0.681,0.90 (0.45-1.79),0.761
5,tobaccoLR,0.87 (0.29-2.62),0.799,1.18 (0.34-4.07),0.789
6,alcoholLR,0.52 (0.25-1.06),0.072,0.69 (0.26-1.84),0.462


In [46]:
import pandas as pd
import statsmodels.api as sm
import numpy as np

# 1. Prepare Age as the only predictor
# This turns categories like '31-40' into numbers the model can use
X_age = pd.get_dummies(df['Age'], prefix='Age', drop_first=True).astype(float)
y = pd.factorize(df['SBP_2lLR'])[0]

# 2. Run Logistic Regression for Age Alone
X_age_const = sm.add_constant(X_age)
model_age = sm.Logit(y, X_age_const).fit()

# 3. Format the Age Results for your table
age_params = model_age.params[1:] # Skip the constant
age_pvalues = model_age.pvalues[1:]
age_conf = np.exp(model_age.conf_int().iloc[1:])
age_ors = np.exp(age_params)

# Create a clean DataFrame for the Age rows
age_results = pd.DataFrame({
    'Variable': age_params.index,
    'Crude OR (95% CI)': [f"{or_val:.2f} ({ci[0]:.2f}-{ci[1]:.2f})" for or_val, ci in zip(age_ors, age_conf.values)],
    'p-value (Crude)': age_pvalues.round(3).values
})

print("--- AGE ALONE (UNIVARIATE) RESULTS ---")
print(age_results)

Optimization terminated successfully.
         Current function value: 0.597590
         Iterations 5
--- AGE ALONE (UNIVARIATE) RESULTS ---
Empty DataFrame
Columns: [Variable, Crude OR (95% CI), p-value (Crude)]
Index: []


In [18]:
import pandas as pd
import statsmodels.api as sm
import numpy as np

# 1. Merge 18-29 and 30-39 because 18-29 has zero '1' outcomes
df['Age_Fixed'] = df['Age'].replace({
    '18-29': '18-39', 
    '30-39': '18-39'
})

# 2. Setup Variables
y = pd.factorize(df['SBP_2lLR'])[0].astype(float)
covariates = ['BMI', 'GenderLR', 'RBG', 'famly_hx_of_htnLR', 'dietary_saltLR', 'tobaccoLR', 'alcoholLR', 'Age_Fixed']

def run_model(cols):
    X = pd.get_dummies(df[cols], drop_first=True, dtype=int)
    X = sm.add_constant(X)
    # Using 'bfgs' to ensure convergence with the new categories
    model = sm.Logit(y, X).fit(method='bfgs', disp=False)
    return model

# 3. Build Table Rows
final_data = []

# Get Adjusted Results first (for all variables)
adj_model = run_model(covariates)
adj_ors = np.exp(adj_model.params)
adj_cis = np.exp(adj_model.conf_int())
adj_ps = adj_model.pvalues

# Loop through each variable to get Crude results
for var in covariates:
    crude_model = run_model([var])
    c_ors = np.exp(crude_model.params)
    c_cis = np.exp(crude_model.conf_int())
    c_ps = crude_model.pvalues
    
    for i in range(1, len(c_ors)):
        var_label = c_ors.index[i]
        
        # Format Crude
        crude_str = f"{c_ors[i]:.2f} ({c_cis.iloc[i,0]:.2f}-{c_cis.iloc[i,1]:.2f})"
        
        # Match with Adjusted values
        adj_str = f"{adj_ors[var_label]:.2f} ({adj_cis.loc[var_label,0]:.2f}-{adj_cis.loc[var_label,1]:.2f})"
        
        final_data.append({
            'Variable': var_label,
            'Crude OR (95% CI)': crude_str,
            'p-value (Crude)': f"{c_ps[i]:.3f}",
            'Adjusted OR (95% CI)': adj_str,
            'p-value (Adjusted)': f"{adj_ps[var_label]:.3f}"
        })

# 4. Display Final Table
final_table = pd.DataFrame(final_data)
print("--- FINAL PUBLICATION TABLE ---")
print(final_table.to_string(index=False))

C:\Users\ajayi\anaconda3\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
C:\Users\ajayi\AppData\Local\Temp\ipykernel_14128\2880138306.py:42: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  crude_str = f"{c_ors[i]:.2f} ({c_cis.iloc[i,0]:.2f}-{c_cis.iloc[i,1]:.2f})"
C:\Users\ajayi\AppData\Local\Temp\ipykernel_14128\2880138306.py:50: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  'p-value (Crude)': f"{c_ps[i]:.3f}",
C:\Users\ajayi\AppData\Local\Temp\ipykernel_14128\28801383

--- FINAL PUBLICATION TABLE ---
              Variable Crude OR (95% CI) p-value (Crude) Adjusted OR (95% CI) p-value (Adjusted)
                   BMI  0.98 (0.92-1.04)           0.440     0.94 (0.87-1.01)              0.102
              GenderLR  0.60 (0.30-1.17)           0.135     0.79 (0.27-2.34)              0.673
                   RBG  1.01 (0.91-1.12)           0.836     1.00 (0.87-1.16)              0.950
     famly_hx_of_htnLR  0.68 (0.19-2.43)           0.554     0.85 (0.19-3.79)              0.836
        dietary_saltLR  0.87 (0.44-1.70)           0.681     1.09 (0.51-2.34)              0.815
             tobaccoLR  0.87 (0.29-2.62)           0.799     1.59 (0.38-6.60)              0.526
             alcoholLR  0.52 (0.25-1.06)           0.072     0.45 (0.15-1.36)              0.156
       Age_Fixed_40-49  0.41 (0.10-1.69)           0.216     0.37 (0.08-1.64)              0.191
       Age_Fixed_50-59  0.23 (0.06-0.89)           0.034     0.22 (0.05-0.90)              0.03

C:\Users\ajayi\AppData\Local\Temp\ipykernel_14128\2880138306.py:42: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  crude_str = f"{c_ors[i]:.2f} ({c_cis.iloc[i,0]:.2f}-{c_cis.iloc[i,1]:.2f})"
C:\Users\ajayi\AppData\Local\Temp\ipykernel_14128\2880138306.py:50: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  'p-value (Crude)': f"{c_ps[i]:.3f}",


In [25]:
import pandas as pd
import statsmodels.api as sm
import numpy as np

# 1. Load the fresh data
df = pd.read_excel("Mass_screening_agbado_april2026_analyze.xlsx")

# 2. Fix the Age Groups (Merging to avoid the '0' cell issue)
df['Age_Fixed'] = df['Age'].replace({
    '18-29': '18-39', 
    '30-39': '18-39'
})

# 3. Define variables
# Ensure your outcome is numeric (1 for High, 0 for Normal)
y = df['SBP_2lLR'].astype(float)

# List all your predictors (Crude and Adjusted)
covariates = [
    'BMI', 'GenderLR', 'RBG', 'famly_hx_of_htnLR', 
    'dietary_saltLR', 'tobaccoLR', 'alcoholLR', 'Age_Fixed'
]

def run_model(cols):
    X = pd.get_dummies(df[cols], drop_first=True, dtype=int)
    X = sm.add_constant(X)
    # Using 'bfgs' solver for better convergence
    return sm.Logit(y, X).fit(method='bfgs', disp=False)

# 4. Generate the Final Publication Table
final_rows = []

# Get Adjusted Model Results (All together)
adj_model = run_model(covariates)
adj_ors, adj_cis, adj_ps = np.exp(adj_model.params), np.exp(adj_model.conf_int()), adj_model.pvalues

# Loop for Crude Model Results (One by one)
for var in covariates:
    crude_model = run_model([var])
    c_ors, c_cis, c_ps = np.exp(crude_model.params), np.exp(crude_model.conf_int()), crude_model.pvalues
    
    for i in range(1, len(c_ors)):
        v_label = c_ors.index[i]
        
        # Format strings for the table
        crude_str = f"{c_ors[i]:.2f} ({c_cis.iloc[i,0]:.2f}-{c_cis.iloc[i,1]:.2f})"
        adj_str = f"{adj_ors[v_label]:.2f} ({adj_cis.loc[v_label,0]:.2f}-{adj_cis.loc[v_label,1]:.2f})"
        
        final_rows.append({
            'Variable': v_label,
            'Crude OR (95% CI)': crude_str,
            'p-value (Crude)': f"{c_ps[i]:.3f}",
            'Adjusted OR (95% CI)': adj_str,
            'p-value (Adjusted)': f"{adj_ps[v_label]:.3f}"
        })

# 5. Show results
publication_table = pd.DataFrame(final_rows)
print("\n--- FINAL PUBLICATION TABLE (REF: 18-39) ---")
print(publication_table.to_string(index=False))

C:\Users\ajayi\anaconda3\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
C:\Users\ajayi\AppData\Local\Temp\ipykernel_14128\876848870.py:46: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  crude_str = f"{c_ors[i]:.2f} ({c_cis.iloc[i,0]:.2f}-{c_cis.iloc[i,1]:.2f})"
C:\Users\ajayi\AppData\Local\Temp\ipykernel_14128\876848870.py:52: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  'p-value (Crude)': f"{c_ps[i]:.3f}",
C:\Users\ajayi\AppData\Local\Temp\ipykernel_14128\876848870.


--- FINAL PUBLICATION TABLE (REF: 18-39) ---
              Variable  Crude OR (95% CI) p-value (Crude) Adjusted OR (95% CI) p-value (Adjusted)
                   BMI   1.02 (0.96-1.09)           0.440     1.06 (0.99-1.14)              0.102
              GenderLR   1.68 (0.85-3.31)           0.135     1.26 (0.43-3.73)              0.673
                   RBG   0.99 (0.89-1.10)           0.836     1.00 (0.87-1.14)              0.950
     famly_hx_of_htnLR   1.47 (0.41-5.23)           0.554     1.17 (0.26-5.19)              0.836
        dietary_saltLR   1.15 (0.59-2.25)           0.681     0.91 (0.43-1.95)              0.815
             tobaccoLR   1.16 (0.38-3.49)           0.799     0.63 (0.15-2.62)              0.526
             alcoholLR   1.93 (0.94-3.93)           0.072     2.24 (0.73-6.82)              0.156
       Age_Fixed_40-49  2.47 (0.59-10.30)           0.216    2.69 (0.61-11.87)              0.191
       Age_Fixed_50-59  4.38 (1.12-17.10)           0.034    4.63 (1.11-

In [27]:
# for logistic regression for a categorical age column

import pandas as pd
import statsmodels.api as sm
import numpy as np

# 1. LOAD DATA
# Make sure the file name matches your edited Excel file
df = pd.read_excel("Mass_screening_agbado_april2026_analyze.xlsx")

# 2. PRE-PROCESSING
# Merge younger groups to ensure we have 'events' in the reference category
df['Age_Fixed'] = df['Age'].replace({'18-29': '18-39', '30-39': '18-39'})

# Ensure target is float (1 = High SBP, 0 = Normal)
y = df['SBP_2lLR'].astype(float)

# Define all independent variables for the Adjusted model
covariates = [
    'BMI', 'GenderLR', 'RBG', 'famly_hx_of_htnLR', 
    'dietary_saltLR', 'tobaccoLR', 'alcoholLR', 'Age_Fixed'
]

# 3. ANALYSIS FUNCTION
def run_logit(cols):
    # drop_first=True makes the first alphabetical category the reference group
    X = pd.get_dummies(df[cols], drop_first=True, dtype=int)
    X = sm.add_constant(X)
    return sm.Logit(y, X).fit(method='bfgs', disp=False)

# 4. BUILD RESULTS TABLE
final_results = []

# Get Adjusted stats (all variables)
adj_model = run_logit(covariates)
a_ors, a_cis, a_ps = np.exp(adj_model.params), np.exp(adj_model.conf_int()), adj_model.pvalues

# Get Crude stats (one variable at a time)
for var in covariates:
    c_model = run_logit([var])
    c_ors, c_cis, c_ps = np.exp(c_model.params), np.exp(c_model.conf_int()), c_model.pvalues
    
    for i in range(1, len(c_ors)): # Skip constant
        v_name = c_ors.index[i]
        
        final_results.append({
            'Variable': v_name,
            'Crude OR (95% CI)': f"{c_ors[i]:.2f} ({c_cis.iloc[i,0]:.2f}-{c_cis.iloc[i,1]:.2f})",
            'p-value (Crude)': f"{c_ps[i]:.3f}",
            'Adjusted OR (95% CI)': f"{a_ors[v_name]:.2f} ({a_cis.loc[v_name,0]:.2f}-{a_cis.loc[v_name,1]:.2f})",
            'p-value (Adjusted)': f"{a_ps[v_name]:.3f}"
        })

# 5. DISPLAY
output_table = pd.DataFrame(final_results)
print("\n--- FINAL PUBLICATION TABLE ---")
print(output_table.to_string(index=False))

# Optional: Save to Excel for easy copy-pasting into Word
# output_table.to_excel('Regression_Results.xlsx', index=False)

C:\Users\ajayi\anaconda3\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
C:\Users\ajayi\AppData\Local\Temp\ipykernel_14128\2132473423.py:46: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  'Crude OR (95% CI)': f"{c_ors[i]:.2f} ({c_cis.iloc[i,0]:.2f}-{c_cis.iloc[i,1]:.2f})",
C:\Users\ajayi\AppData\Local\Temp\ipykernel_14128\2132473423.py:47: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  'p-value (Crude)': f"{c_ps[i]:.3f}",
C:\Users\ajayi\AppData\Local\Temp\ipykernel_1412


--- FINAL PUBLICATION TABLE ---
              Variable  Crude OR (95% CI) p-value (Crude) Adjusted OR (95% CI) p-value (Adjusted)
                   BMI   1.02 (0.96-1.09)           0.440     1.06 (0.99-1.14)              0.102
              GenderLR   1.68 (0.85-3.31)           0.135     1.26 (0.43-3.73)              0.673
                   RBG   0.99 (0.89-1.10)           0.836     1.00 (0.87-1.14)              0.950
     famly_hx_of_htnLR   1.47 (0.41-5.23)           0.554     1.17 (0.26-5.19)              0.836
        dietary_saltLR   1.15 (0.59-2.25)           0.681     0.91 (0.43-1.95)              0.815
             tobaccoLR   1.16 (0.38-3.49)           0.799     0.63 (0.15-2.62)              0.526
             alcoholLR   1.93 (0.94-3.93)           0.072     2.24 (0.73-6.82)              0.156
       Age_Fixed_40-49  2.47 (0.59-10.30)           0.216    2.69 (0.61-11.87)              0.191
       Age_Fixed_50-59  4.38 (1.12-17.10)           0.034    4.63 (1.11-19.29)       

In [28]:
# for logistic regression for a categorical age column

import pandas as pd
import statsmodels.api as sm
import numpy as np

# 1. LOAD DATA
# Make sure the file name matches your edited Excel file
df = pd.read_excel("Mass_screening_agbado_april2026_analyze.xlsx")

# 2. PRE-PROCESSING
# Merge younger groups to ensure we have 'events' in the reference category
df['Age_Fixed'] = df['Age'].replace({'18-29': '18-39', '30-39': '18-39'})

# Ensure target is float (1 = High SBP, 0 = Normal)
y = df['DBP_2lLR'].astype(float)

# Define all independent variables for the Adjusted model
covariates = [
    'BMI', 'GenderLR', 'RBG', 'famly_hx_of_htnLR', 
    'dietary_saltLR', 'tobaccoLR', 'alcoholLR', 'Age_Fixed'
]

# 3. ANALYSIS FUNCTION
def run_logit(cols):
    # drop_first=True makes the first alphabetical category the reference group
    X = pd.get_dummies(df[cols], drop_first=True, dtype=int)
    X = sm.add_constant(X)
    return sm.Logit(y, X).fit(method='bfgs', disp=False)

# 4. BUILD RESULTS TABLE
final_results = []

# Get Adjusted stats (all variables)
adj_model = run_logit(covariates)
a_ors, a_cis, a_ps = np.exp(adj_model.params), np.exp(adj_model.conf_int()), adj_model.pvalues

# Get Crude stats (one variable at a time)
for var in covariates:
    c_model = run_logit([var])
    c_ors, c_cis, c_ps = np.exp(c_model.params), np.exp(c_model.conf_int()), c_model.pvalues
    
    for i in range(1, len(c_ors)): # Skip constant
        v_name = c_ors.index[i]
        
        final_results.append({
            'Variable': v_name,
            'Crude OR (95% CI)': f"{c_ors[i]:.2f} ({c_cis.iloc[i,0]:.2f}-{c_cis.iloc[i,1]:.2f})",
            'p-value (Crude)': f"{c_ps[i]:.3f}",
            'Adjusted OR (95% CI)': f"{a_ors[v_name]:.2f} ({a_cis.loc[v_name,0]:.2f}-{a_cis.loc[v_name,1]:.2f})",
            'p-value (Adjusted)': f"{a_ps[v_name]:.3f}"
        })

# 5. DISPLAY
output_table = pd.DataFrame(final_results)
print("\n--- FINAL PUBLICATION TABLE ---")
print(output_table.to_string(index=False))

# Optional: Save to Excel for easy copy-pasting into Word
# output_table.to_excel('Regression_Results.xlsx', index=False)

C:\Users\ajayi\anaconda3\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
C:\Users\ajayi\AppData\Local\Temp\ipykernel_14128\2584594286.py:48: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  'Crude OR (95% CI)': f"{c_ors[i]:.2f} ({c_cis.iloc[i,0]:.2f}-{c_cis.iloc[i,1]:.2f})",
C:\Users\ajayi\AppData\Local\Temp\ipykernel_14128\2584594286.py:49: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  'p-value (Crude)': f"{c_ps[i]:.3f}",
C:\Users\ajayi\AppData\Local\Temp\ipykernel_1412


--- FINAL PUBLICATION TABLE ---
              Variable  Crude OR (95% CI) p-value (Crude) Adjusted OR (95% CI) p-value (Adjusted)
                   BMI   1.08 (1.01-1.15)           0.024     1.12 (1.03-1.21)              0.005
              GenderLR   1.88 (0.91-3.87)           0.088     2.33 (0.74-7.36)              0.148
                   RBG   1.05 (0.95-1.16)           0.324     1.08 (0.95-1.22)              0.248
     famly_hx_of_htnLR  3.16 (0.91-10.91)           0.069    2.85 (0.66-12.33)              0.161
        dietary_saltLR   0.98 (0.47-2.05)           0.957     0.94 (0.41-2.13)              0.875
             tobaccoLR   1.66 (0.54-5.08)           0.372     1.13 (0.28-4.55)              0.865
             alcoholLR   2.01 (0.95-4.29)           0.069     1.26 (0.40-3.98)              0.689
       Age_Fixed_40-49  3.17 (0.60-16.72)           0.175    2.70 (0.51-14.35)              0.243
       Age_Fixed_50-59 10.56 (2.23-50.01)           0.003    9.19 (1.93-43.82)       

C:\Users\ajayi\AppData\Local\Temp\ipykernel_14128\2584594286.py:49: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  'p-value (Crude)': f"{c_ps[i]:.3f}",


In [29]:
# for logistic regression for a categorical age column

import pandas as pd
import statsmodels.api as sm
import numpy as np

# 1. LOAD DATA
# Make sure the file name matches your edited Excel file
df = pd.read_excel("Mass_screening_agbado_april2026_analyze.xlsx")

# 2. PRE-PROCESSING
# Merge younger groups to ensure we have 'events' in the reference category
df['Age_Fixed'] = df['Age'].replace({'18-29': '18-39', '30-39': '18-39'})

# Ensure target is float (1 = High SBP, 0 = Normal)
y = df['RBG_LR'].astype(float)

# Define all independent variables for the Adjusted model
covariates = [
    'BMI', 'GenderLR', 'SBP', 'DBP', 'Family_history_of_DMLR', 
     'tobaccoLR', 'alcoholLR', 'Age_Fixed'
]

# 3. ANALYSIS FUNCTION
def run_logit(cols):
    # drop_first=True makes the first alphabetical category the reference group
    X = pd.get_dummies(df[cols], drop_first=True, dtype=int)
    X = sm.add_constant(X)
    return sm.Logit(y, X).fit(method='bfgs', disp=False)

# 4. BUILD RESULTS TABLE
final_results = []

# Get Adjusted stats (all variables)
adj_model = run_logit(covariates)
a_ors, a_cis, a_ps = np.exp(adj_model.params), np.exp(adj_model.conf_int()), adj_model.pvalues

# Get Crude stats (one variable at a time)
for var in covariates:
    c_model = run_logit([var])
    c_ors, c_cis, c_ps = np.exp(c_model.params), np.exp(c_model.conf_int()), c_model.pvalues
    
    for i in range(1, len(c_ors)): # Skip constant
        v_name = c_ors.index[i]
        
        final_results.append({
            'Variable': v_name,
            'Crude OR (95% CI)': f"{c_ors[i]:.2f} ({c_cis.iloc[i,0]:.2f}-{c_cis.iloc[i,1]:.2f})",
            'p-value (Crude)': f"{c_ps[i]:.3f}",
            'Adjusted OR (95% CI)': f"{a_ors[v_name]:.2f} ({a_cis.loc[v_name,0]:.2f}-{a_cis.loc[v_name,1]:.2f})",
            'p-value (Adjusted)': f"{a_ps[v_name]:.3f}"
        })

# 5. DISPLAY
output_table = pd.DataFrame(final_results)
print("\n--- FINAL PUBLICATION TABLE ---")
print(output_table.to_string(index=False))

# Optional: Save to Excel for easy copy-pasting into Word
# output_table.to_excel('Regression_Results.xlsx', index=False)

C:\Users\ajayi\anaconda3\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
C:\Users\ajayi\AppData\Local\Temp\ipykernel_14128\3324813467.py:48: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  'Crude OR (95% CI)': f"{c_ors[i]:.2f} ({c_cis.iloc[i,0]:.2f}-{c_cis.iloc[i,1]:.2f})",
C:\Users\ajayi\AppData\Local\Temp\ipykernel_14128\3324813467.py:49: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  'p-value (Crude)': f"{c_ps[i]:.3f}",
C:\Users\ajayi\AppData\Local\Temp\ipykernel_1412


--- FINAL PUBLICATION TABLE ---
              Variable Crude OR (95% CI) p-value (Crude) Adjusted OR (95% CI) p-value (Adjusted)
                   BMI  1.07 (1.01-1.13)           0.031     1.04 (0.98-1.12)              0.202
              GenderLR  0.56 (0.28-1.13)           0.103     0.89 (0.33-2.43)              0.822
                   SBP  1.01 (0.99-1.02)           0.363     1.01 (0.99-1.04)              0.299
                   DBP  1.01 (0.99-1.03)           0.443     0.98 (0.94-1.02)              0.286
Family_history_of_DMLR  2.88 (0.98-8.46)           0.055    3.32 (0.99-11.16)              0.052
             tobaccoLR  0.56 (0.17-1.81)           0.333     1.37 (0.36-5.31)              0.645
             alcoholLR  0.53 (0.25-1.13)           0.101     0.53 (0.19-1.49)              0.230
       Age_Fixed_40-49 5.19 (1.88-14.32)           0.001    5.58 (1.90-16.41)              0.002
       Age_Fixed_50-59  3.21 (1.15-8.91)           0.026    3.86 (1.27-11.76)              0.0